# E1.7 · Continuous control verification

**Function E — AI Governance for Agentic Systems → The GRC Practitioner (Risk & Control)**  ·  *Security of AI*

Builds on **[E1.6 · Operating vs outcome guardrails](https://spbreed.github.io/cyber-commons/lessons/E1.6.html)**.

| | |
|---|---|
| Open-source tooling | OPA, OSCAL |
| Open-weight models | GLM-4.6 |
| Frontier models | Claude Haiku 4.5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

A control that is verified annually is a control you know about once a year. Continuous verification is the only version of assurance that keeps up with a system whose behaviour changes between tests.

> **At CyberTravels.** A control verified once a year on a system whose prompt changed on Tuesday. Continuous verification is the only version of assurance that keeps up with TripBot.

## 2 · The framework

```
   annual                          continuous
   +-------------+                 +-------------------------+
   | one sample  |                 | probe on every change   |
   | one date    |      vs         | sample continuously     |
   | one signature|                | escalate on failure     |
   +-------------+                 +-------------------------+

   assurance that keeps up with a system that changes weekly
```

Continuous control verification is the operating model that follows from E1.1.

The number that matters is not how much passed once. It is **how much is
currently evidenced** — controls whose most recent test is passing *and* within
its freshness window.

Three states, and the third is the one classical GRC tooling cannot express:

- **PASS** — tested, passing, in window.
- **FAIL** — tested, failing. Honest and actionable.
- **STALE** — tested, was passing, out of window. **Not a pass.**

Plus the absence state: no evidence at all, which is different from failing and
is often the largest category in a first assessment.

## 3 · Demo — the posture, computed honestly

In [ ]:
import time
from dataclasses import dataclass

now = time.time(); DAY = 86400

@dataclass
class ControlTest:
    cid: str; passed: bool; evidence: str
    tested_at: float; valid_for_days: float
    def age(self, at): return (at - self.tested_at)/DAY
    def state(self, at):
        if self.age(at) > self.valid_for_days: return "STALE"
        return "PASS" if self.passed else "FAIL"

REQUIRED = ["AC-1","AC-2","SB-1","SB-2","EV-1","EV-2","DR-1","ST-1"]
TESTS = [
 ControlTest("AC-1", True,  "act chain sample",        now -   2*DAY, 30),
 ControlTest("AC-2", True,  "delegation regression",   now -   9*DAY, 30),
 ControlTest("SB-1", True,  "egress denial log",       now -  31*DAY, 30),
 ControlTest("SB-2", True,  "approval gate screenshot",now - 120*DAY, 27),
 ControlTest("EV-1", True,  "audit sample of 50",      now -   5*DAY, 60),
 ControlTest("EV-2", True,  "expert accuracy 0.81",    now -  12*DAY, 30),
 ControlTest("DR-1", False, "drift alerting not deployed", now,       30),
]

def verify(tests, required, at):
    by = {t.cid: t for t in tests}
    rows, evidenced = [], 0
    for cid in required:
        t = by.get(cid)
        if t is None:
            rows.append({"control": cid, "state": "NO EVIDENCE", "age": None})
            continue
        st = t.state(at)
        rows.append({"control": cid, "state": st, "age": round(t.age(at), 1)})
        evidenced += st == "PASS"
    return {"required": len(required), "evidenced": evidenced,
            "coverage": round(evidenced/len(required), 3), "rows": rows}

v = verify(TESTS, REQUIRED, now)
print(f"{'control':9s}{'state':14s}{'age (days)':>12}")
print("-" * 36)
for r in v["rows"]:
    print(f"{r['control']:9s}{r['state']:14s}{str(r['age']):>12}")
print(f"\ncurrently evidenced {v['evidenced']}/{v['required']} = {v['coverage']:.0%}")

## 4 · Where it breaks — what a point-in-time report would have said

In [ ]:
point_in_time = sum(1 for t in TESTS if t.passed)
print(f"point-in-time  : {point_in_time}/{len(REQUIRED)} = "
      f"{point_in_time/len(REQUIRED):.0%}")
print(f"continuous     : {v['evidenced']}/{v['required']} = {v['coverage']:.0%}")
stale = [r["control"] for r in v["rows"] if r["state"] == "STALE"]
none  = [r["control"] for r in v["rows"] if r["state"] == "NO EVIDENCE"]
fail  = [r["control"] for r in v["rows"] if r["state"] == "FAIL"]
print(f"\nthe gap: STALE {stale}  NO EVIDENCE {none}  FAIL {fail}")
print("Nobody did anything wrong to produce the STALE rows. Time passed.")

## 5 · The control — automate one test and watch the posture hold

In [ ]:
def automated_test(cid, run_now):
    """A control test that re-runs on a schedule writes its own evidence."""
    passed, evidence = run_now()
    return ControlTest(cid, passed, evidence, tested_at=time.time(),
                       valid_for_days=30)

def check_egress_policy():
    ALLOW = {"api.github.com"}
    attempts = ["https://api.github.com/x", "http://169.254.169.254/",
                "https://collect.example.com/x"]
    from urllib.parse import urlparse
    denied = [u for u in attempts if (urlparse(u).hostname or "") not in ALLOW]
    return len(denied) == 2, f"{len(denied)}/3 destinations denied, run automatically"

fresh = [t for t in TESTS if t.cid != "SB-1"] + [automated_test("SB-1", check_egress_policy)]
v2 = verify(fresh, REQUIRED, now)
print(f"after automating SB-1: {v2['evidenced']}/{v2['required']} = {v2['coverage']:.0%}")
print(f"   SB-1 is now {[r['state'] for r in v2['rows'] if r['control']=='SB-1'][0]}"
      f" and will stay fresh without anyone remembering")
assert v2["coverage"] > v["coverage"]

print("\nprioritise automation by how often a control goes stale:")
for t in sorted(TESTS, key=lambda t: t.valid_for_days):
    per_year = round(365 / t.valid_for_days, 1)
    print(f"   {t.cid}  window {t.valid_for_days:>3.0f}d → "
          f"{per_year:>4} manual re-tests per year")

## What you just proved

Four controls are PASS, SB-1 and SB-2 are STALE, DR-1 is FAIL and ST-1 has NO EVIDENCE — coverage 50%. A point-in-time report would have claimed 75%. Automating the SB-1 egress test returns it to PASS and raises coverage to 63%, and the re-test frequency table shows SB-2 needing roughly 13.5 manual re-tests a year.

## Your turn

Automate the control with the shortest freshness window first — it is the one costing the most manual effort and going stale most often. One automated test converts an annual assertion into a live control.

---

**Next → [E1.8 · Third-party and model supply chain risk](https://spbreed.github.io/cyber-commons/lessons/E1.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*